# RUSA AI ↔ RusaModels Codespaces API smoke test

Run this notebook from Google Colab to verify that a Codespaces-hosted RusaModels Model Server is reachable over HTTPS. It checks the authenticated model catalog, chat completion, and 384-dimensional multilingual-E5 embeddings.

## Before you run
1. Start the RusaModels **CPU Model Server** Codespaces profile and wait for chat and embedding runtimes to become ready.
2. Forward only Model Server port `8000`; never forward llama.cpp `8080` or embedding `8081` directly.
3. If connecting from ordinary Colab, make the Model Server forwarded port publicly reachable over HTTPS and protect it with a dedicated, short-lived RusaModels bearer token. A private Codespaces port requires GitHub-authenticated access that a normal Colab runtime does not automatically have.
4. Keep the token in Colab's hidden input or Secret Manager. Do not save it in notebook source or outputs.

This is an API connectivity test, not a production RUSA AI deployment. Keep RAG disabled until RUSA AI migration `009_knowledge_embedding_384.sql` is applied and the tenant corpus is re-embedded.

In [ ]:
import getpass
import math
from urllib.parse import urlsplit

import requests

BASE_URL = input("RusaModels HTTPS base URL (must end with /openai/v1): ").strip().rstrip("/")
parsed = urlsplit(BASE_URL)
assert parsed.scheme == "https", "Use the HTTPS Codespaces forwarded URL"
assert parsed.path.rstrip("/").endswith("/openai/v1"), "Base URL must include /openai/v1"
assert parsed.hostname and not parsed.username and not parsed.password
assert not parsed.query and not parsed.fragment

API_TOKEN = getpass.getpass("RusaModels bearer token (hidden): ").strip()
assert len(API_TOKEN) >= 32, "Token is missing or too short"
HEADERS = {"Authorization": f"Bearer {API_TOKEN}", "Content-Type": "application/json"}
SESSION = requests.Session()
SESSION.trust_env = False
print("HTTPS endpoint format validated; token captured without displaying it.")

In [ ]:
response = SESSION.get(f"{BASE_URL}/models", headers=HEADERS, timeout=(10, 30))
response.raise_for_status()
model_ids = {item["id"] for item in response.json().get("data", [])}
print("Models exposed by RusaModels:", sorted(model_ids))
assert "rusa-general-4b" in model_ids, "Chat model is not configured"
assert "rusa-embedding-small" in model_ids, "Embedding model is not configured"
print("Model catalog check: PASS")

In [ ]:
chat_payload = {
    "model": "rusa-general-4b",
    "messages": [
        {"role": "system", "content": "Return a concise answer."},
        {"role": "user", "content": "Reply with: RusaModels chat connection is working."},
    ],
    "temperature": 0,
    "max_tokens": 96,
    "stream": False,
}
response = SESSION.post(
    f"{BASE_URL}/chat/completions",
    headers=HEADERS,
    json=chat_payload,
    timeout=(10, 180),
)
response.raise_for_status()
chat_text = response.json()["choices"][0]["message"]["content"]
assert isinstance(chat_text, str) and chat_text.strip()
print(chat_text)
print("Chat completion check: PASS")

In [ ]:
def embed_one(text, input_type):
    response = SESSION.post(
        f"{BASE_URL}/embeddings",
        headers=HEADERS,
        json={
            "model": "rusa-embedding-small",
            "input": [text],
            "input_type": input_type,
            "encoding_format": "float",
        },
        timeout=(10, 120),
    )
    response.raise_for_status()
    vector = response.json()["data"][0]["embedding"]
    assert len(vector) == 384, f"Expected 384 dimensions, got {len(vector)}"
    norm = math.sqrt(sum(float(value) ** 2 for value in vector))
    assert math.isclose(norm, 1.0, rel_tol=1e-3, abs_tol=1e-3), f"Expected normalized vector, got norm {norm}"
    return vector

passage_vector = embed_one("RUSA AI uses tenant-scoped retrieval and agent orchestration.", "passage")
query_vector = embed_one("How does RUSA AI retrieve knowledge?", "query")
print({"passage_dimensions": len(passage_vector), "query_dimensions": len(query_vector)})
print("E5 query/passage embedding check: PASS")

## RUSA AI environment values

The following cell prepares environment variables for RUSA AI code running in this Colab session. It does not install or start the RUSA AI worker, connect a PostgreSQL database, or enable RAG. Configure the same values in the real worker's secret manager when deploying it.

In [ ]:
import os

os.environ["RUSA_MODEL_BASE_URL"] = BASE_URL
os.environ["RUSA_MODEL_NAME"] = "rusa-general-4b"
os.environ["RUSA_MODEL_API_KEY"] = API_TOKEN
os.environ["RUSA_EMBEDDING_BASE_URL"] = BASE_URL
os.environ["RUSA_EMBEDDING_MODEL"] = "rusa-embedding-small"
os.environ["RUSA_EMBEDDING_API_KEY"] = API_TOKEN
os.environ["RUSA_EMBEDDING_DIMENSIONS"] = "384"
os.environ["RUSA_EMBEDDING_INPUT_TYPE_ENABLED"] = "true"
os.environ["RUSA_EMBEDDING_TIMEOUT"] = "120"
os.environ["RUSA_RAG_ENABLED"] = "false"

print({
    "RUSA_MODEL_BASE_URL": os.environ["RUSA_MODEL_BASE_URL"],
    "RUSA_MODEL_NAME": os.environ["RUSA_MODEL_NAME"],
    "RUSA_EMBEDDING_MODEL": os.environ["RUSA_EMBEDDING_MODEL"],
    "RUSA_EMBEDDING_DIMENSIONS": os.environ["RUSA_EMBEDDING_DIMENSIONS"],
    "RUSA_EMBEDDING_INPUT_TYPE_ENABLED": os.environ["RUSA_EMBEDDING_INPUT_TYPE_ENABLED"],
    "RUSA_RAG_ENABLED": os.environ["RUSA_RAG_ENABLED"],
    "RUSA_MODEL_API_KEY": "<set in memory; not displayed>",
})

## Cutover gate

A successful smoke test proves only that the Model Server endpoints are reachable from this Colab runtime. Before setting `RUSA_RAG_ENABLED=true` in RUSA AI:
- Merge and deploy the RusaModels runtime change.
- Apply RUSA AI migration `009_knowledge_embedding_384.sql` through the database migration process.
- Backfill each tenant using `python scripts/reembed_knowledge.py --tenant-id <tenant-id> --collection agent` first without `--apply`, then with `--apply` after review.
- Run retrieval evaluation and tenant-isolation tests.

Codespaces is an ephemeral development host. Do not treat this notebook or a forwarded development port as a production availability guarantee.